In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np

# Load data (adjust filename if needed)
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Combine train and test for consistent preprocessing
full = pd.concat([train, test], sort=False, ignore_index=True)

# Basic Cleaning and Feature Engineering
# 1. Handle missing values
for col in full.columns:
    if full[col].dtype == 'object':
        # Fill categorical NA with the mode
        full[col] = full[col].fillna(full[col].mode()[0])
    else:
        # Fill numerical NA with the median
        full[col] = full[col].fillna(full[col].median())

# 2. Encode categorical variables
full = pd.get_dummies(full)

# 3. Example: Feature Transformation
# Log-transform the target variable for normality
if 'SalePrice' in full:
    full['SalePrice'] = np.log1p(full['SalePrice'])

# Split back into train and test
train_cleaned = full.loc[:len(train)-1].copy()
test_cleaned = full.loc[len(train):].copy()

# Check results
print(train_cleaned.shape, test_cleaned.shape)
train_cleaned.head()

# (you can now proceed to modeling)


In [ ]:
# Split features and target
X = train_cleaned.drop('SalePrice', axis=1)
y = train_cleaned['SalePrice']

# Split train-validation
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Metrics
from sklearn.metrics import mean_squared_error

def print_rmse(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name} RMSE: {rmse:.4f}")

# 1. Linear Regression
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)
print_rmse(y_val, y_pred_lr, 'Linear Regression')

# 2. Neural Net
from sklearn.neural_network import MLPRegressor
mlp = MLPRegressor(hidden_layer_sizes=(128,64), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_val)
print_rmse(y_val, y_pred_mlp, 'Neural Net')

# 3. Decision Tree
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_val)
print_rmse(y_val, y_pred_dt, 'Decision Tree')

# 4. Random Forest
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)
print_rmse(y_val, y_pred_rf, 'Random Forest')

# 5. Gradient Boosting
from sklearn.ensemble import GradientBoostingRegressor
gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_val)
print_rmse(y_val, y_pred_gbr, 'Gradient Boosting')

# 6. XGBoost
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, tree_method='hist')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_val)
print_rmse(y_val, y_pred_xgb, 'XGBoost')

# 7. LightGBM
from lightgbm import LGBMRegressor
lgb = LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1)
lgb.fit(X_train, y_train)
y_pred_lgb = lgb.predict(X_val)
print_rmse(y_val, y_pred_lgb, 'LightGBM')

# 8. CatBoost
from catboost import CatBoostRegressor
cat = CatBoostRegressor(n_estimators=100, random_state=42, verbose=0)
cat.fit(X_train, y_train)
y_pred_cat = cat.predict(X_val)
print_rmse(y_val, y_pred_cat, 'CatBoost')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# X = your features DataFrame, y = your target Series (presumably already processed)
# X, y = train_cleaned.drop('SalePrice', axis=1), train_cleaned['SalePrice']

# For demo purposes, if you haven't loaded data yet:
# df_cols = [f'feat_{i}' for i in range(15)]
# np.random.seed(42)
# X = pd.DataFrame(np.random.randn(100, 15), columns=df_cols)
# y = np.random.randn(100) * 50000 + 180000

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_results = []
fitted_models = {}

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)
lr_rmse = np.sqrt(mean_squared_error(y_val, y_pred_lr))
model_results.append(('Linear Regression', lr_rmse))
fitted_models['Linear Regression'] = lr

# Neural Net
mlp = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=200, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_val)
mlp_rmse = np.sqrt(mean_squared_error(y_val, y_pred_mlp))
model_results.append(('Neural Net', mlp_rmse))
fitted_models['Neural Net'] = mlp

# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_val)
dt_rmse = np.sqrt(mean_squared_error(y_val, y_pred_dt))
model_results.append(('Decision Tree', dt_rmse))
fitted_models['Decision Tree'] = dt

# Random Forest
rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_pred_rf))
model_results.append(('Random Forest', rf_rmse))
fitted_models['Random Forest'] = rf

# Gradient Boosting
gbr = GradientBoostingRegressor(n_estimators=50, random_state=42)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_val)
gbr_rmse = np.sqrt(mean_squared_error(y_val, y_pred_gbr))
model_results.append(('Gradient Boosting', gbr_rmse))
fitted_models['Gradient Boosting'] = gbr

# ---- Results Table ----
results_df = pd.DataFrame(model_results, columns=['Model', 'Validation_RMSE']).sort_values('Validation_RMSE')
print(results_df)

# ---- Best Model Selection ----
best_idx = results_df['Validation_RMSE'].idxmin()
best_model_name = results_df.iloc[0]['Model']
print(f"Best Model: {best_model_name}")

# ---- RMSE Bar Chart ----
plt.figure(figsize=(8,5))
plt.barh(results_df['Model'], results_df['Validation_RMSE'], color='skyblue')
plt.xlabel('Validation RMSE')
plt.title('Model Comparison (Lower is Better)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# ---- Feature Importance for Tree Models ----
if best_model_name in ['Random Forest', 'Gradient Boosting', 'Decision Tree']:
    best_model = fitted_models[best_model_name]
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        features = X
